xgboost, svc, knn, lda, qda, guassian nb, rf

In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv('/Users/tliu/Desktop/Erdos Project/3_Player_Data_Generation/match_data_20_tourns_modified.csv')
data.head()

,player1,player2,best_of,player1_elo,player2_elo,elo_match_win_rate,elo_frame_win_rate,p1_matches_played,p1_matches_won,p1_frames_played,...,p1_frames_played_3_years,p1_frames_won_3_years,p2_frames_played_1_year,p2_frames_won_1_year,p2_frames_played_3_years,p2_frames_won_3_years,score1,score2,match_result,win_percentage
0,Long Zehuang,Haydon Pinhey,7,1328,1158,0.719210,0.604679,78,39,443,...,284,147,121,57,347,168,4,3,0.0,0.571429
1,Wang Yuchen,Andrew Pagett,7,1162,1162,0.500000,0.500000,100,37,626,...,158,89,144,65,417,170,4,1,0.0,0.800000
2,Ben Mertens,Daniel Womersley,7,1262,1109,0.699444,0.594476,107,50,642,...,471,240,86,40,180,89,1,4,1.0,0.200000
3,Paul Deaville,Jimmy White,7,1096,1141,0.438735,0.471905,32,17,158,...,126,66,85,31,384,153,3,4,1.0,0.428571
4,Alexander Ursenbacher,Mostafa Dorgham,7,1302,1031,0.821305,0.663180,286,129,1728,...,400,196,108,41,154,53,4,1,0.0,0.800000


In [3]:
#Add more features
data['p1_frames_win_rate'] = data['p1_frames_won']/ data['p1_frames_played']
data['p2_frames_win_rate'] = data['p2_frames_won']/ data['p2_frames_played']

data['p1_matches_win_rate'] = data['p1_matches_won']/ data['p1_matches_played']
data['p2_matches_win_rate'] = data['p2_matches_won']/ data['p2_matches_played']

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1914 entries, 0 to 1913
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   player1                   1914 non-null   object 
 1   player2                   1914 non-null   object 
 2   best_of                   1914 non-null   int64  
 3   player1_elo               1914 non-null   int64  
 4   player2_elo               1914 non-null   int64  
 5   elo_match_win_rate        1914 non-null   float64
 6   elo_frame_win_rate        1914 non-null   float64
 7   p1_matches_played         1914 non-null   int64  
 8   p1_matches_won            1914 non-null   int64  
 9   p1_frames_played          1914 non-null   int64  
 10  p1_frames_won             1914 non-null   int64  
 11  p2_matches_played         1914 non-null   int64  
 12  p2_matches_won            1914 non-null   int64  
 13  p2_frames_played          1914 non-null   int64  
 14  p2_frame

In [5]:
#p1_matches_win_rate and p2_matches_win_rate both have missing values
#We will fill them with 0.5
data.fillna(0.5, inplace = True)

In [6]:
#Train test split
from sklearn.model_selection import train_test_split
data_train, data_test = train_test_split(data, 
                                        test_size = 0.2,
                                        shuffle = True,
                                        random_state=216)

In [7]:
#Create two dictionaries to record the scores.
win_perc_pred_scores = {}

match_result_pred_scores = {}

In [8]:
#Import metrics from sklearn.metrics
from sklearn.metrics import accuracy_score, root_mean_squared_error

In [9]:
#Create predictors and targets for training and test set
result_train = data_train['match_result']
win_perc_train = data_train['win_percentage']
#exclude players' names and match results.
X_train = data_train.drop(['match_result', 'win_percentage', 'player1', 'player2', 'score1', 'score2'], axis = 1)


result_test = data_test['match_result']
win_perc_test = data_test['win_percentage']
X_test = data_test.drop(['match_result', 'win_percentage', 'player1', 'player2', 'score1', 'score2'], axis = 1)

## Model1: Random Forest

In [10]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

n_estinamtors = list(range(100, 800, 100))

m_depth = list(range(3, 9))

n_1, n_2, m_1, m_2 = 0,0,0,0

score1_best = 100
score2_best = 0

for n in n_estinamtors: 
    print(f'Random Forest: n_estimators = {n}')
    for m in m_depth:
        RFR = RandomForestRegressor(n_estimators = n, max_depth = m)

        RFR.fit(X_train, win_perc_train)
        pred = RFR.predict(X_test)
        score1 = root_mean_squared_error(pred, win_perc_test)
        if score1 < score1_best:
            score1_best = score1
            n_1 = n
            m_1 = m

        RFC = RandomForestClassifier(n_estimators = n, max_depth = m)
        RFC.fit(X_train, result_train)
        pred2 = RFC.predict(X_test)
        score2 = accuracy_score(pred2, result_test)

        if score2 > score2_best:
            score2_best = score2
            n_2 = n
            m_2 = m

#Record the best scores
win_perc_pred_scores[f'RFR(n_estimators = {n}, max_depth = {m})'] = score1_best
match_result_pred_scores[f'RFC(n_estimators = {n}, max_depth = {m})'] = score2_best

Random Forest: n_estimators = 100
Random Forest: n_estimators = 200
Random Forest: n_estimators = 300
Random Forest: n_estimators = 400
Random Forest: n_estimators = 500
Random Forest: n_estimators = 600
Random Forest: n_estimators = 700


## Random Forest with PCA

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
n_estinamtors = list(range(100, 800, 100))

m_depth = list(range(3, 9))

n_1, n_2, m_1, m_2 = 0,0,0,0

score1_best = 100
score2_best = 0

for n in n_estinamtors: 
    print(f'Random Forest: n_estimators = {n}')
    for m in m_depth:
        RFR = Pipeline([('pca', PCA(3)),
                         ("RFR", RandomForestRegressor(n_estimators = n, max_depth = m))])


        RFR.fit(X_train, win_perc_train)
        pred = RFR.predict(X_test)
        score1 = root_mean_squared_error(pred, win_perc_test)
        if score1 < score1_best:
            score1_best = score1
            n_1 = n
            m_1 = m

        RFC = Pipeline([('pca', PCA(3)),
                         ("RFC", RandomForestClassifier(n_estimators = n, max_depth = m))])
        RFC.fit(X_train, result_train)
        pred2 = RFC.predict(X_test)
        score2 = accuracy_score(pred2, result_test)

        if score2 > score2_best:
            score2_best = score2
            n_2 = n
            m_2 = m

#Record the best scores
win_perc_pred_scores[f'RFR(n_estimators = {n}, max_depth = {m}) with PCA(3)'] = score1_best
match_result_pred_scores[f'RFC(n_estimators = {n}, max_depth = {m}) with PCA(3)'] = score2_best

Random Forest: n_estimators = 100
Random Forest: n_estimators = 200
Random Forest: n_estimators = 300
Random Forest: n_estimators = 400
Random Forest: n_estimators = 500
Random Forest: n_estimators = 600
Random Forest: n_estimators = 700


In [14]:
print(win_perc_pred_scores)
print(match_result_pred_scores)

{'RFR(n_estimators = 700, max_depth = 8)': 0.27604879894953377, 'RFR(n_estimators = 700, max_depth = 8) with PCA(3)': 0.2910418219483309}
{'RFC(n_estimators = 700, max_depth = 8)': 0.6710182767624021, 'RFC(n_estimators = 700, max_depth = 8) with PCA(3)': 0.6109660574412533}
